# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import time
from datetime import datetime
from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, RetrievalEvaluation
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, VectorStore
from lib.memory import LongTermMemory, MemorySearchResult, MemoryFragment
from lib.evaluation import AgentEvaluator, TestCase
from lib import dashboard_logs
from typing import List, Optional, Dict, Any


In [3]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

#CONSTANTS
load_dotenv('.env')

CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

CHROMADB_PATH = "chromadb"
COLLECTION_NAME = "udaplay"

# Vector DB
vector_store_manager = VectorStoreManager(openai_api_key=CHROMA_OPENAI_API_KEY)
game_store: VectorStore = vector_store_manager.get_or_create_store(COLLECTION_NAME)
long_term_memory = LongTermMemory(db=vector_store_manager)

USER_ID = "default_user"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game



In [5]:
@tool
def retrieve_game(query:str, n_results:int):
    """
    Semantic search: Finds the most relevant games in the vector DB.

    args:
    - query: a question about the game industry.
    - n_results: number of results to return.

    Returns a list of document strings, where each element contains:
    - Platform
    - Name
    - YearOfRelease
    - Description
    """
    # Query the database

    results = game_store.query(
        query_texts=[query], 
        n_results=n_results
        )
        
    retrieved_docs = results['documents'][0]
    return retrieved_docs

#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [7]:
@tool
def evaluate_retrieval(question:str, retrieved_docs:list) -> RetrievalEvaluation:
    """
    Evaluate whether the retrieved documents are sufficient to answer the user's question.
    Call this after retrieve_game to decide if you can answer from internal knowledge or need web search.

    Args:
        question: The user's original question.
        retrieved_docs: List of document strings returned by retrieve_game.

    Returns:
        useful: True if the documents are sufficient to answer; False if web search is needed.
        description: Explanation of why the documents are or are not sufficient.
    """
    #Initialize the Judge LLM
    judge_llm = LLM(
    model="gpt-4o-mini", 
    temperature=0.0,
    api_key=OPENAI_API_KEY
    )
    
    # Turn docs into a readable string for the judge LLM
    docs_text = "\n\n".join(f"- {doc}" for doc in retrieved_docs)

    #2. Build the Messages
    system_msg = SystemMessage(
        content=(
            "You are a strict evaluator for a retrieval-augmented generation (RAG) system. "
            "Given a question and a set of retrieved documents, your job is ONLY to decide "
            "whether these documents are sufficient to answer the question, and to explain why.\n\n"
            "You must respond using the structured schema you have been given."
        )
    )

    user_msg = UserMessage(
        content=(
            f"Question:\n{question}\\n"
            f"Retrieved docuemnts:\n{docs_text}\n\n"
            "Decide if these documents are enough to answer the question correctly."
        )
    )

    ai_message = judge_llm.invoke(
        input=[system_msg, user_msg],
        response_format=RetrievalEvaluation
    )

    evaluation: RetrievalEvaluation = ai_message.content
    return evaluation

#### Game Web Search Tool

In [8]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 



In [9]:
@tool
def game_web_search(question:str) -> Dict:
    """
    Search the web for information about video games (release dates, platforms, publishers, etc.).
    Use when internal knowledge from retrieve_game is insufficient or when evaluate_retrieval says the documents are not useful.

    Args:
        question: The search query (e.g., a question about games, release dates, or platforms).

    Returns:
        answer: Tavily's direct answer if available; results: list of search results with titles and snippets.
    """
    #Initialize the Tavily Client
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    
    # Perform the search
    search_result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }
    
    return formatted_results

#### Memory Store And Retrieval Tool 

In [10]:
@tool
def retrieve_memory(question: str, limit: int = 3) -> Dict:
    """
    Retrieve relevant long-term memories for the current user.

    args:
    - question: the user's current question.
    - limit: max number of memory fragments to return.

    Returns:
    - fragments: list of strings (memory contents)
    - metadata: distances or any other scores
    """
    result = long_term_memory.search(
        query_text=question,
        owner=USER_ID,
        limit=limit,
        namespace="default",
    )

    return {
        "fragments": [f.content for f in result.fragments],
        "metadata": result.metadata,
    }

In [11]:
@tool
def store_memory(content: str, namespace: str = "default") -> Dict:
    """
    Store a new long-term memory for the current user.

    args:
    - content: the text to remember.
    - namespace: optional logical grouping.

    Returns:
    - success: bool
    """
    fragment = MemoryFragment(
        content=content,
        owner=USER_ID,
        namespace=namespace,
    )
    long_term_memory.register(fragment)
    return {"success": True}

### Agent

In [12]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [
    retrieve_game, 
    evaluate_retrieval, 
    game_web_search, 
    retrieve_memory, 
    store_memory
    ]

# instructions = (
# "You are a research agent for the video game industry."
# "You can answer multistep questions by sequentially calling functions for:"
# "- Answer questions using internal knowledge (RAG)"
# "- call retrieve_memory at the beginning when prior context may matter"
# "- Run the evaluate_retrieval tool to see if the retrieved information is enought to answer"
# "- Search the web when the internal knowledge is insufficient using the game web search"
# "- After answering, if it discovers a user preference or durable fact, call store_memory with a short text summary."
# "You follow a pattern of of Thought and Action. "
# "Create a plan of execution: "
# "- Use Thought to describe your thoughts about the question you have been asked. "
# "- Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly."
# "When you think it's over, return the answer "
# "Never try to respond directly if the question needs a tool. "
# "But if you don't have a tool available, you can respond directly. "
# f"The actions you have are the Tools: {tools}. \n"
# )

# I used GPT5.4 to help me imporve the prompt based on the initial prompt
# 

improved_instructions = f"""
You are UdaPlay, a research agent for the video game industry.

You must answer factual questions by following the required tool workflow below. This workflow is mandatory.

REQUIRED WORKFLOW FOR FACTUAL QUESTIONS
1. First call `retrieve_game`.
2. Second call `evaluate_retrieval` using:
   - the original user question
   - the documents returned by `retrieve_game`
3. Read the result of `evaluate_retrieval`.
4. If `evaluate_retrieval.useful` is True:
   - do NOT call `game_web_search`
   - answer using the retrieved internal documents
5. If `evaluate_retrieval.useful` is False:
   - then call `game_web_search`
   - answer using the best available evidence from search results

HARD RULES
- You are NOT allowed to call `game_web_search` immediately after `retrieve_game`.
- You MUST call `evaluate_retrieval` before any web search.
- `game_web_search` is allowed only if `evaluate_retrieval` explicitly says the retrieved documents are not sufficient.
- For ordinary factual game questions, `evaluate_retrieval` is mandatory.
- Do not skip `evaluate_retrieval` even if you think you already know the answer.

DECISION POLICY
- Treat `evaluate_retrieval` as the decision gate between internal knowledge and web search.
- If the retrieval evaluation says the documents are enough, stop using tools and answer.
- If the retrieval evaluation says the documents are not enough, use web search exactly once unless a second search is clearly necessary.
- Prefer the shortest valid workflow.

MEMORY RULES
- Use `retrieve_memory` only if the current question depends on prior user-specific context, preferences, or earlier conversation history.
- Do not use `retrieve_memory` for normal factual benchmark questions.
- Use `store_memory` only for durable user-specific preferences or personal facts learned during the conversation.
- Never store public game facts in memory.

ANSWER RULES
- After the required workflow is complete, provide a direct final answer.
- If the answer came from internal retrieval, say it is based on internal game data.
- If the answer required web search, say it was verified with web search.
- If evidence is incomplete or ambiguous, say that clearly instead of guessing.

VALID TOOL SEQUENCES
- Internal-answer path:
  `retrieve_game` -> `evaluate_retrieval` -> final answer
- Web-fallback path:
  `retrieve_game` -> `evaluate_retrieval` -> `game_web_search` -> final answer

INVALID TOOL SEQUENCES
- `retrieve_game` -> `game_web_search`
- `game_web_search` without `evaluate_retrieval`
- direct final answer before retrieval for factual questions

Available tools: {tools}
"""

In [13]:

agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=improved_instructions,
)

In [14]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

q1 = "When Pokémon Gold and Silver was released?"
q2 = "Which one was the first 3D platformer Mario game?"
q3 = "Was Mortal Kombat X realeased for Playstation 5?"


In [15]:
run_object = agent.invoke(
    query=q1,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="\nYou are UdaPlay, a research agent for the video game industry.\n\nYou must answer factual questions by following the required tool workflow below. This workflow is mandatory.\n\nREQUIRED WORKFLOW FOR FACTUAL QUESTIONS\n1. First call `retrieve_game`.\n2. Second call `evaluate_retrieval` using:\n   - the original user question\n   - the documents returned by `retrieve_game`\n3. Read the result of `evaluate_retrieval`.\n4. If `evaluate_retrieval.useful` is True:\n   - do NOT call `game_web_search`\n   - answer using the retrieved internal documents\n5. If `evaluate_retrieval.useful` is False:\n   - then call `game_web_search`\n   - answer using the best available evidence from search results\n\nHARD RULES\n- You are NOT allowed to call `game_web_search` immediately after `retrieve_game`.\n- You MUST call `evaluate_retrieval` before any web search.\n- `game_web_search` is allowed only if `evaluate_retrieval` explicitly says the retrieved documents a

In [16]:
run_object = agent.invoke(
    query=q2,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="\nYou are UdaPlay, a research agent for the video game industry.\n\nYou must answer factual questions by following the required tool workflow below. This workflow is mandatory.\n\nREQUIRED WORKFLOW FOR FACTUAL QUESTIONS\n1. First call `retrieve_game`.\n2. Second call `evaluate_retrieval` using:\n   - the original user question\n   - the documents returned by `retrieve_game`\n3. Read the result of `evaluate_retrieval`.\n4. If `evaluate_retrieval.useful` is True:\n   - do NOT call `game_web_search`\n   - answer using the retrieved internal documents\n5. If `evaluate_retrieval.useful` is False:\n   - then call `game_web_search`\n   - answer using the best available evidence from search results\n\nHARD RULES\n- You are NOT allowed to call `game_web_search` immediately after `retrieve_game`.\n- You MUST call `evaluate_retrieval` before any web search.\n- `game_web_search` is allowed only if `evaluate_retrieval` explicitly says the retrieved documents a

In [17]:
run_object = agent.invoke(
    query=q3,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="\nYou are UdaPlay, a research agent for the video game industry.\n\nYou must answer factual questions by following the required tool workflow below. This workflow is mandatory.\n\nREQUIRED WORKFLOW FOR FACTUAL QUESTIONS\n1. First call `retrieve_game`.\n2. Second call `evaluate_retrieval` using:\n   - the original user question\n   - the documents returned by `retrieve_game`\n3. Read the result of `evaluate_retrieval`.\n4. If `evaluate_retrieval.useful` is True:\n   - do NOT call `game_web_search`\n   - answer using the retrieved internal documents\n5. If `evaluate_retrieval.useful` is False:\n   - then call `game_web_search`\n   - answer using the best available evidence from search results\n\nHARD RULES\n- You are NOT allowed to call `game_web_search` immediately after `retrieve_game`.\n- You MUST call `evaluate_retrieval` before any web search.\n- `game_web_search` is allowed only if `evaluate_retrieval` explicitly says the retrieved documents a

### Evaluation

This section benchmarks the agent on a fixed set of test cases using three complementary views:
- final-response evaluation,
- single-step tool evaluation,
- and full trajectory evaluation.

Each benchmark run uses an isolated `session_id` and `USER_ID` so prior memory does not leak across cases.

In [18]:
evaluator = AgentEvaluator()

benchmark_cases = [
    TestCase(
        id="gold_silver_release",
        description="Check whether the agent answers a local release-date question using retrieval.",
        user_query=q1,
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Pokemon Gold and Silver were first released in Japan in 1999.",
        max_steps=5,
    ),
    TestCase(
        id="first_3d_mario",
        description="Check whether the agent identifies the first 3D Mario platformer from the local dataset.",
        user_query=q2,
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Super Mario 64 was the first 3D Mario platformer.",
        max_steps=5,
    ),
    TestCase(
        id="mkx_ps5",
        description="Check whether the agent falls back to web search for a platform/version question that is not well covered locally.",
        user_query=q3,
        expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
        reference_answer="Mortal Kombat X was not released for PlayStation 5.",
        max_steps=6,
    ),
    TestCase(
        id="ambiguous_case",
        description="Check how the agent handles a broad and ambiguous release question.",
        user_query="When was Pokemon released?",
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer=None,
        max_steps=5,
    ),
    TestCase(
        id="edge_case",
        description="Check whether the agent uses web fallback for a stable platform/version question outside the local dataset.",
        user_query="Was The Elder Scrolls V: Skyrim released for Nintendo Switch?",
        expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
        reference_answer="Yes. The Elder Scrolls V: Skyrim was released for Nintendo Switch.",
        max_steps=6,
    ),
]

In [19]:
def get_final_response_text(run):
    final_state = run.get_final_state() or {}
    messages = final_state.get("messages", [])

    for message in reversed(messages):
        if isinstance(message, AIMessage) and message.content:
            return message.content

    return ""


def get_tool_names(messages):
    tool_names = []

    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            tool_names.extend(tool_call.function.name for tool_call in message.tool_calls)

    return tool_names


def run_benchmark_case(test_case):
    global USER_ID

    original_user_id = USER_ID
    USER_ID = f"eval_{test_case.id}"
    session_id = f"eval_{test_case.id}"

    if session_id in agent.memory.get_all_sessions():
        agent.reset_session(session_id)
    else:
        agent.memory.create_session(session_id)

    try:
        start_time = time.perf_counter()
        run = agent.invoke(query=test_case.user_query, session_id=session_id)
        execution_time = time.perf_counter() - start_time

        final_state = run.get_final_state() or {}
        messages = final_state.get("messages", [])
        final_response = get_final_response_text(run)
        tool_names = get_tool_names(messages)
        total_tokens = final_state.get("total_tokens", 0)

        final_eval = evaluator.evaluate_final_response(
            test_case=test_case,
            agent_response=final_response,
            execution_time=execution_time,
            total_tokens=total_tokens,
        )
        step_eval = evaluator.evaluate_single_step(messages, test_case.expected_tools)
        trajectory_eval = evaluator.evaluate_trajectory(test_case, run)

        return {
            "test_case": test_case,
            "run": run,
            "final_response": final_response,
            "tool_names": tool_names,
            "final_eval": final_eval,
            "step_eval": step_eval,
            "trajectory_eval": trajectory_eval,
            "execution_time": execution_time,
            "total_tokens": total_tokens,
        }
    finally:
        USER_ID = original_user_id


In [20]:
evaluation_results = []
benchmark_run_id = dashboard_logs.create_benchmark_run_id()

for case in benchmark_cases:
    result = run_benchmark_case(case)
    evaluation_results.append(result)
    session_id = f"eval_{result['test_case'].id}"
    ts = datetime.utcnow().isoformat() + "Z"
    dashboard_logs.log_agent_run_from_run(
        result["run"],
        session_id=session_id,
        query=result["test_case"].user_query,
        execution_time_sec=result["execution_time"],
        source="eval",
        benchmark_run_id=benchmark_run_id,
        case_id=result["test_case"].id,
    )
    dashboard_logs.log_eval_case(
        benchmark_run_id=benchmark_run_id,
        case_id=result["test_case"].id,
        run_id=result["run"].run_id,
        timestamp=ts,
        source="eval",
        query=result["test_case"].user_query,
        tools_used=result["tool_names"],
        final_score=result["final_eval"].overall_score,
        trajectory_score=result["trajectory_eval"].overall_score,
        step_tool_correct=result["step_eval"].tool_interaction.correct_tool_selected,
        task_completed=result["trajectory_eval"].task_completion.task_completed,
        execution_time=result["execution_time"],
        total_tokens=result["total_tokens"],
        feedback=result["trajectory_eval"].feedback,
    )

summary_rows = []
for result in evaluation_results:
    row = {
        "case_id": result["test_case"].id,
        "query": result["test_case"].user_query,
        "tools_used": result["tool_names"],
        "final_score": result["final_eval"].overall_score,
        "trajectory_score": result["trajectory_eval"].overall_score,
        "step_tool_correct": result["step_eval"].tool_interaction.correct_tool_selected,
        "task_completed": result["trajectory_eval"].task_completion.task_completed,
        "execution_time": round(result["execution_time"], 3),
        "total_tokens": result["total_tokens"],
        "feedback": result["trajectory_eval"].feedback,
    }
    summary_rows.append(row)

ts_summary = datetime.utcnow().isoformat() + "Z"
dashboard_logs.log_eval_summary(
    benchmark_run_id=benchmark_run_id,
    timestamp=ts_summary,
    total_cases=len(summary_rows),
    mean_final_score=sum(r["final_score"] for r in summary_rows) / len(summary_rows) if summary_rows else None,
    mean_trajectory_score=sum(r["trajectory_score"] for r in summary_rows) / len(summary_rows) if summary_rows else None,
    task_completed_count=sum(1 for r in summary_rows if r["task_completed"]),
    total_tokens=sum(r["total_tokens"] for r in summary_rows),
)

for row in summary_rows:
    print(f"Case: {row['case_id']}")
    print(f"  Query: {row['query']}")
    print(f"  Tools used: {row['tools_used']}")
    print(f"  Final score: {row['final_score']:.2f}")
    print(f"  Trajectory score: {row['trajectory_score']:.2f}")
    print(f"  Step tool correct: {row['step_tool_correct']}")
    print(f"  Task completed: {row['task_completed']}")
    print(f"  Execution time: {row['execution_time']}s")
    print(f"  Total tokens: {row['total_tokens']}")
    print(f"  Feedback: {row['feedback']}")
    print('-' * 80)

summary_rows

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


/var/folders/04/chwb3n9s59j8f4451k4gvy800000gn/T/ipykernel_5300/756804380.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat() + "Z"


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


/var/folders/04/chwb3n9s59j8f4451k4gvy800000gn/T/ipykernel_5300/756804380.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts_summary = datetime.utcnow().isoformat() + "Z"


[{'case_id': 'gold_silver_release',
  'query': 'When Pokémon Gold and Silver was released?',
  'tools_used': ['retrieve_game', 'evaluate_retrieval', 'game_web_search'],
  'final_score': 1.0,
  'trajectory_score': 0.75,
  'step_tool_correct': False,
  'task_completed': False,
  'execution_time': 11.488,
  'total_tokens': 7781,
  'feedback': "Trajectory: 8 steps, Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search'], Expected: ['retrieve_game', 'evaluate_retrieval']"},
 {'case_id': 'first_3d_mario',
  'query': 'Which one was the first 3D platformer Mario game?',
  'tools_used': ['retrieve_game', 'evaluate_retrieval', 'game_web_search'],
  'final_score': 1.0,
  'trajectory_score': 0.75,
  'step_tool_correct': False,
  'task_completed': False,
  'execution_time': 10.853,
  'total_tokens': 7593,
  'feedback': "Trajectory: 8 steps, Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search'], Expected: ['retrieve_game', 'evaluate_retrieval']"},
 {'case_id': 'mkx_ps

In [21]:
failing_results = [
    result
    for result in evaluation_results
    if (
        result["final_eval"].overall_score < 0.75
        or result["trajectory_eval"].overall_score < 0.75
        or result["step_eval"].tool_interaction.correct_tool_selected is False
    )
]

if not failing_results:
    print("No weak runs found with the current benchmark threshold.")
else:
    for result in failing_results:
        test_case = result["test_case"]
        print(f"Case: {test_case.id}")
        print(f"  Expected tools: {test_case.expected_tools}")
        print(f"  Actual tools: {result['tool_names']}")
        print(f"  Final response: {result['final_response']}")
        print(f"  Final evaluation feedback: {result['final_eval'].feedback}")
        print(f"  Trajectory evaluation feedback: {result['trajectory_eval'].feedback}")
        print('-' * 80)

Case: gold_silver_release
  Expected tools: ['retrieve_game', 'evaluate_retrieval']
  Actual tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
  Final response: Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000. This information was verified with web search results.
  Final evaluation feedback: The agent successfully answered the user's query about the release dates of Pokémon Gold and Silver, providing specific dates for both Japan and North America. The format of the response is clear and informative, and it follows the implicit instruction to provide accurate information based on retrieval. Therefore, all criteria are met.
  Trajectory evaluation feedback: Trajectory: 8 steps, Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search'], Expected: ['retrieve_game', 'evaluate_retrieval']
--------------------------------------------------------------------------------
Case: first_3d_mario
  Expected 

### (Optional) Advanced

In [22]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes